# Featured Quotes for Artists

For each artist, pick the non-artwork artist-sourced point with the smallest `distance_to_center` at a fixed n value (same approach used for cluster featured quotes)

In [11]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

In [15]:
ar_clusters_path = Path("../../data/colab/ar_clusters.json")
output_path = Path("../../data/addl/featured_quotes.csv")

with ar_clusters_path.open(encoding="utf-8") as f:
    ar_clusters = json.load(f)

# pick the smallest n as the most stable/broad grouping
min_n_entry = min(ar_clusters, key=lambda e: e["n"])
print(f"Using n={min_n_entry['n']}")

Using n=8


In [16]:
def _is_artwork(val):
    if isinstance(val, bool):
        return val
    if isinstance(val, str):
        return val.strip().lower() in {"true", "1", "yes"}
    return bool(val)


# collect all non-artwork artist-sourced points for this n, keyed by artist_id
artist_candidates: dict[int, list[dict]] = {}
for cluster in min_n_entry["clusters"]:
    for point in cluster["data"]:
        if _is_artwork(point.get("is_artwork")):
            continue
        artist_id = point.get("artist_id")
        if artist_id is None:
            continue
        artist_candidates.setdefault(artist_id, []).append(point)

# for each artist, pick the point with the smallest distance_to_center
rows = []
for artist_id in sorted(artist_candidates):
    candidates = artist_candidates[artist_id]
    selected = min(
        candidates,
        key=lambda p: (
            float(p["distance_to_center"]) if p.get("distance_to_center") is not None else float("inf"),
            str(p.get("source_idx") or ""),
            p.get("point") if p.get("point") is not None else float("inf"),
        ),
    )
    rows.append({
        "artist_id":          artist_id,
        "source_idx":         selected.get("source_idx"),
        "point":              selected.get("point"),
        "distance_to_center": selected.get("distance_to_center"),
        "text":               selected.get("text"),
    })

featured_quotes = pd.DataFrame(rows, columns=["artist_id", "source_idx", "point", "distance_to_center", "text"])
print(f"{len(featured_quotes)} artists")
display(featured_quotes)

30 artists


,artist_id,source_idx,point,distance_to_center,text
0,0,AT53,0,0.562450,"So, I’m still a struggling artist—at least in ..."
1,1,AT01,4,0.515885,Navigating this stuff allows me to investigate...
2,2,AT11,9,0.487654,I was also thinking about the way Asian Americ...
3,3,AT18,17,0.377722,"If somebody is mixed-race, we assume they’re m..."
4,4,AT27,28,0.410198,"That’s a funny question, because it assumes th..."
5,5,AT36,34,0.486047,I don’t think of myself as Japanese. I think o...
6,6,AT46,37,0.473399,"It also suggests the life before me, which is ..."
7,7,AT62,45,0.469344,Does this mean an artist should not represent ...
8,8,AT68,53,0.558580,"I think that’s where I discovered my language,..."
9,9,AT77,60,0.450414,I keep circling back to thinking about identit...


In [17]:
display(featured_quotes)

,artist_id,source_idx,point,distance_to_center,text
0,0,AT53,0,0.562450,"So, I’m still a struggling artist—at least in ..."
1,1,AT01,4,0.515885,Navigating this stuff allows me to investigate...
2,2,AT11,9,0.487654,I was also thinking about the way Asian Americ...
3,3,AT18,17,0.377722,"If somebody is mixed-race, we assume they’re m..."
4,4,AT27,28,0.410198,"That’s a funny question, because it assumes th..."
5,5,AT36,34,0.486047,I don’t think of myself as Japanese. I think o...
6,6,AT46,37,0.473399,"It also suggests the life before me, which is ..."
7,7,AT62,45,0.469344,Does this mean an artist should not represent ...
8,8,AT68,53,0.558580,"I think that’s where I discovered my language,..."
9,9,AT77,60,0.450414,I keep circling back to thinking about identit...


In [10]:
featured_quotes[["artist_id", "source_idx", "point"]].to_csv(output_path, index=False)
print(f"Wrote {output_path}")

Wrote ../../data/addl/featured_quotes.csv
